In [1]:
import sys
from pathlib import Path

# If notebook is inside notebooks/memory-techniques/
project_root = Path.cwd().resolve().parents[1]

print(f"Adding project root to sys.path: {project_root}")
sys.path.insert(0, str(project_root))

from utils.env_loader import load_environment
_ = load_environment()

Adding project root to sys.path: /Users/kanderaolaxminarasimharao/Downloads/MyAgents-Git/MyAgents


In [4]:
import copy
from collections import deque
from openai import OpenAI

class SlidingWindowMemory:
    """Sliding window memory that keeps the last *k* messages."""

    def __init__(
        self,
        window_size: int = 10,
        model: str = "gpt-4.1-mini",
        system_prompt: str | None = None,
        max_tokens: int = 1024,
    ):
        self.client = OpenAI()
        self.model = model
        self.system_prompt = system_prompt
        self.max_tokens = max_tokens

        # The sliding window - a deque with a fixed max length
        self.window_size = window_size
        self.messages: deque[dict] = deque(maxlen=window_size)

        # Track all messages for analysis (not sent to the LLM)
        self.full_history: list[dict] = []
        self.turn_token_usage: list[dict] = []

    # ── Chat ─────────────────────────────────────────────────────
    def chat(self, user_input: str) -> str:
        """Send a message; only the last *k* messages are sent to the LLM."""
        user_msg = {"role": "user", "content": user_input}
        self.messages.append(user_msg)
        self.full_history.append(user_msg)

        kwargs = dict(
            model=self.model,
            input=list(self.messages),  # deque -> list for the API
            max_output_tokens=self.max_tokens,
        )
        if self.system_prompt:
            kwargs["instructions"] = self.system_prompt

        response = self.client.responses.create(**kwargs)
        assistant_text = response.output_text

        assistant_msg = {"role": "assistant", "content": assistant_text}
        self.messages.append(assistant_msg)
        self.full_history.append(assistant_msg)

        self.turn_token_usage.append({
            "turn": len(self.turn_token_usage) + 1,
            "input_tokens": response.usage.input_tokens if response.usage else None,
            "output_tokens": response.usage.output_tokens if response.usage else None,
            "total_tokens": response.usage.total_tokens if response.usage else None,
            "window_msgs": len(self.messages),
        })

        return assistant_text

    # ── Inspection helpers ───────────────────────────────────────
    def get_window(self) -> list[dict]:
        """Return the current window contents (what the LLM sees)."""
        return list(self.messages)

    def get_full_history(self) -> list[dict]:
        """Return all messages ever sent, including evicted ones."""
        return copy.deepcopy(self.full_history)

    @property
    def evicted_count(self) -> int:
        """Number of messages that have been dropped from the window."""
        return len(self.full_history) - len(self.messages)

    def clear(self) -> None:
        self.messages.clear()
        self.full_history.clear()
        self.turn_token_usage.clear()

    def __repr__(self) -> str:
        return (
            f"SlidingWindowMemory(window={len(self.messages)}/{self.window_size}, "
            f"total={len(self.full_history)} messages, "
            f"evicted={self.evicted_count})"
        )

print("✓ SlidingWindowMemory class defined")


✓ SlidingWindowMemory class defined


In [5]:
mem = SlidingWindowMemory(
    window_size=6,  # 3 full turns (user + assistant each)
    system_prompt="You are a concise assistant. Reply in 1-2 sentences.",
)

conversation = [
    "My name is Alice and I'm a pilot.",           # turn 1 - plants a fact
    "I fly Boeing 737s for a regional airline.",    # turn 2 - plants another fact
    "What's the weather like in Seattle today?",   # turn 3 - unrelated filler
    "What's my name and what do I do for work?",   # turn 4 - recall test (turn 1 may be evicted)
]

for msg in conversation:
    print(f"👤 User:  {msg}")
    reply = mem.chat(msg)
    print(f"🤖 Agent: {reply}")
    print(f"   📊 Window: {len(mem.messages)}/{mem.window_size} msgs | Evicted: {mem.evicted_count}")
    print()

👤 User:  My name is Alice and I'm a pilot.
🤖 Agent: Nice to meet you, Alice! How can I assist you today?
   📊 Window: 2/6 msgs | Evicted: 0

👤 User:  I fly Boeing 737s for a regional airline.
🤖 Agent: That's impressive, Alice! Do you need any information or assistance related to the Boeing 737?
   📊 Window: 4/6 msgs | Evicted: 0

👤 User:  What's the weather like in Seattle today?
🤖 Agent: I currently can't provide real-time weather updates. Please check a reliable weather website or app for Seattle's current conditions.
   📊 Window: 6/6 msgs | Evicted: 0

👤 User:  What's my name and what do I do for work?
🤖 Agent: Your name is Alice, and you fly Boeing 737s for a regional airline.
   📊 Window: 6/6 msgs | Evicted: 2



In [6]:
for turn in mem.turn_token_usage:
    print(
        f"Turn {turn['turn']}: "
        f"input={turn['input_tokens']}, "
        f"output={turn['output_tokens']}, "
        f"total={turn['total_tokens']}, "
        f"window_msgs={turn['window_msgs']}"
    )

Turn 1: input=34, output=15, total=49, window_msgs=2
Turn 2: input=67, output=20, total=87, window_msgs=4
Turn 3: input=102, output=24, total=126, window_msgs=6
Turn 4: input=131, output=18, total=149, window_msgs=6


In [7]:
# Inspect what the LLM currently sees vs. what it has forgotten
print("=== Messages IN the window (LLM can see) ===")
for i, msg in enumerate(mem.get_window()):
    role = "USER" if msg["role"] == "user" else "ASST"
    preview = msg["content"][:70] + ("..." if len(msg["content"]) > 70 else "")
    print(f"  [{i}] {role}: {preview}")

print(f"\n=== Evicted messages (LLM cannot see): {mem.evicted_count} ===" )
evicted = mem.get_full_history()[:mem.evicted_count]
for i, msg in enumerate(evicted):
    role = "USER" if msg["role"] == "user" else "ASST"
    preview = msg["content"][:70] + ("..." if len(msg["content"]) > 70 else "")
    print(f"  [{i}] {role}: {preview}")

=== Messages IN the window (LLM can see) ===
  [0] USER: I fly Boeing 737s for a regional airline.
  [1] ASST: That's impressive, Alice! Do you need any information or assistance re...
  [2] USER: What's the weather like in Seattle today?
  [3] ASST: I currently can't provide real-time weather updates. Please check a re...
  [4] USER: What's my name and what do I do for work?
  [5] ASST: Your name is Alice, and you fly Boeing 737s for a regional airline.

=== Evicted messages (LLM cannot see): 2 ===
  [0] USER: My name is Alice and I'm a pilot.
  [1] ASST: Nice to meet you, Alice! How can I assist you today?


In [8]:
import re

FACTS = [
    ("My name is Elena.", "elena"),
    ("I was born in Prague.", "prague"),
    ("My favorite color is teal.", "teal"),
    ("I have two cats named Salt and Pepper.", "salt"),
    ("I work as a data scientist at a biotech company.", "data scientist"),
]

FILLER_MESSAGES = [
    "Tell me a fun fact about octopuses.",
    "What's the capital of Mongolia?",
    "How does photosynthesis work in one sentence?",
    "What year was the Eiffel Tower built?",
    "Name a famous mathematician.",
]

RECALL_QUESTIONS = [
    ("What is my name?", "elena"),
    ("Where was I born?", "prague"),
    ("What is my favorite color?", "teal"),
    ("What are my cats' names?", "salt"),
    ("What do I do for work?", "data scientist"),
]



In [9]:

def run_recall_eval(window_size: int) -> dict:
    """Run the recall eval for a given window size and return scores."""
    mem = SlidingWindowMemory(
        window_size=window_size,
        system_prompt="You are a helpful assistant. Answer concisely.",
    )

    # Phase 1: Plant facts (5 turns = 10 messages)
    for fact, _ in FACTS:
        mem.chat(fact)

    # Phase 2: Filler turns to push old facts out (5 turns = 10 messages)
    for filler in FILLER_MESSAGES:
        mem.chat(filler)

    # Phase 3: Ask recall questions and check answers
    results = {}
    for question, keyword in RECALL_QUESTIONS:
        answer = mem.chat(question)
        recalled = keyword.lower() in answer.lower()
        results[keyword] = {
            "question": question,
            "answer": answer[:100],
            "recalled": recalled,
        }

    score = sum(1 for r in results.values() if r["recalled"])
    return {
        "window_size": window_size,
        "score": score,
        "total": len(RECALL_QUESTIONS),
        "pct": score / len(RECALL_QUESTIONS) * 100,
        "details": results,
        "total_input_tokens": sum(t["input_tokens"] for t in mem.turn_token_usage),
    }



In [10]:

# Test multiple window sizes
WINDOW_SIZES = [4, 8, 12, 20]
eval_results = []

for k in WINDOW_SIZES:
    print(f"Running eval with window_size={k}...", end=" ")
    result = run_recall_eval(k)
    eval_results.append(result)
    print(f"Recall: {result['score']}/{result['total']} ({result['pct']:.0f}%)")

print("\nDone!")

Running eval with window_size=4... Recall: 0/5 (0%)
Running eval with window_size=8... Recall: 0/5 (0%)
Running eval with window_size=12... Recall: 1/5 (20%)
Running eval with window_size=20... Recall: 4/5 (80%)

Done!
